<a href="https://colab.research.google.com/github/VakitiSriVarsha/Gen-AI/blob/main/Sentence_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu chromadb

In [ ]:
passages = [
    "Machine learning is a field of artificial intelligence that uses statistical techniques to give computer systems the ability to learn from data.",
    "Deep learning is a subset of machine learning that uses neural networks with many layers.",
    "Python is a popular programming language known for its simplicity and readability.",
    "FAISS is a library developed by Facebook AI Research for efficient similarity search and clustering of dense vectors.",
    "Chroma is an open-source embedding database designed for easy RAG applications.",
    "Transformers are neural network architectures based on self-attention mechanisms.",
    "Natural language processing enables computers to understand and generate human language.",
    "Wikipedia is a free online encyclopedia written and maintained by volunteers."
]

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(passages)
print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (8, 384)


In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Total vectors in index:", index.ntotal)

Total vectors in index: 8


In [ ]:
query = "What is deep neural networks?"

query_embedding = model.encode([query]).astype("float32")

k = 3
distances, indices = index.search(query_embedding, k)

print("Query:", query)
print("\nTop Results:")
for i in indices[0]:
    print("-", passages[i])

Query: What is deep neural networks?

Top Results:
- Deep learning is a subset of machine learning that uses neural networks with many layers.
- Machine learning is a field of artificial intelligence that uses statistical techniques to give computer systems the ability to learn from data.
- Transformers are neural network architectures based on self-attention mechanisms.


In [ ]:
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.create_collection(name="my_docs")

collection.add(
    documents=passages,
    ids=[f"doc_{i}" for i in range(len(passages))]
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 103MiB/s]


In [ ]:
results = collection.query(
    query_texts=["What is artificial intelligence?"],
    n_results=3
)

for doc in results["documents"][0]:
    print("-", doc)


- Machine learning is a field of artificial intelligence that uses statistical techniques to give computer systems the ability to learn from data.
- Deep learning is a subset of machine learning that uses neural networks with many layers.
- Natural language processing enables computers to understand and generate human language.
